# Notebook 05 — Multilingual Visual RAG

Phase 2 opener. We take the ColPali pipeline from nb03 and point it at a **Chinese financial report**, then compare it against a traditional `Tesseract OCR + multilingual text embedding` baseline. The point isn't that ColPali wins by a small margin — the point is that *the failure modes are completely different*, and the OCR baseline fails in ways that are very expensive to fix in production.

## The thesis

ColPali's backbone (PaliGemma) was pretrained on a multilingual corpus. Once you skip the OCR step, **adding a new language is free**: same code, same model, different PDF.

Traditional OCR pipelines, by contrast, need:
- A language-specific OCR engine (or a multilingual one with quality drops)
- Layout-aware post-processing (Chinese has no word boundaries, Japanese has vertical layout, etc.)
- A multilingual embedding model that handles all your target languages well

Each of those is a separate failure surface. We'll see one of them break in front of you.

## Setup — get a Chinese financial PDF

Drop a Chinese financial report into `../data/sample_pdfs/` named `cn_financial.pdf`. Recommended public sources:

1. **Tencent (00700.HK) Annual Report 中文版** — `https://www.tencent.com/zh-cn/investors.html` → 年度報告
2. **Alibaba (9988.HK) ESG 报告** — `https://www.alibabagroup.com/document` → 选中文版
3. **中国人民银行金融稳定报告** — `http://www.pbc.gov.cn/` → 政策研究 → 金融稳定报告
4. **国家统计局年度公报** — `http://www.stats.gov.cn/`

Pick whichever has charts and tables — that's where ColPali's advantage is most visible.

In [ ]:
from pathlib import Path

PDF_PATH = Path("../data/sample_pdfs/cn_financial.pdf")
assert PDF_PATH.exists(), (
    f"Drop a Chinese financial PDF at {PDF_PATH.resolve()} "
    f"(see the markdown cell above for recommended sources)."
)
print(f"Found: {PDF_PATH} ({PDF_PATH.stat().st_size / 1e6:.1f} MB)")

In [ ]:
# Render. Cap pages so the lab stays interactive — ColPali on 200 pages is slow.
from pdf2image import convert_from_path
import matplotlib.pyplot as plt

MAX_PAGES = 30
pages = convert_from_path(str(PDF_PATH), dpi=150, last_page=MAX_PAGES)
print(f"Rendered {len(pages)} pages.")

fig, axes = plt.subplots(1, 3, figsize=(12, 5))
for ax, p, i in zip(axes, pages[:3], range(3)):
    ax.imshow(p); ax.set_title(f"page {i}"); ax.set_axis_off()
plt.tight_layout(); plt.show()

## Path A — ColPali (no OCR)

Same code as nb03. The model file is identical — we don't switch to a Chinese variant. *That's the point.*

In [ ]:
import torch
from transformers import ColPaliForRetrieval, ColPaliProcessor

device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

MODEL_NAME = "vidore/colpali-v1.3-hf"
model = ColPaliForRetrieval.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map=device
).eval()
processor = ColPaliProcessor.from_pretrained(MODEL_NAME)

@torch.no_grad()
def colpali_embed_pages(pil_pages, batch_size=4):
    out = []
    for i in range(0, len(pil_pages), batch_size):
        batch = pil_pages[i : i + batch_size]
        inputs = processor(images=batch, return_tensors="pt").to(device)
        for emb in model(**inputs).embeddings:
            out.append(emb.cpu())
    return out

@torch.no_grad()
def colpali_embed_query(query: str):
    inputs = processor(text=[query], return_tensors="pt").to(device)
    return model(**inputs).embeddings[0].cpu()

def colpali_maxsim(q_emb, page_emb):
    sim = q_emb.float() @ page_emb.float().T
    return float(sim.max(dim=1).values.sum())

def colpali_search(query: str, page_embs, k=3):
    q = colpali_embed_query(query)
    scored = sorted(
        ((colpali_maxsim(q, pe), idx) for idx, pe in enumerate(page_embs)),
        reverse=True,
    )
    return scored[:k]

print("Embedding pages with ColPali...")
page_embeddings = colpali_embed_pages(pages)
print(f"Done. {len(page_embeddings)} pages indexed.")

## Path B — Tesseract OCR + multilingual text embedding

The traditional pipeline: extract text per page with Tesseract, embed with a multilingual sentence transformer, search. We use `paraphrase-multilingual-MiniLM-L12-v2` (~120 MB, supports 50+ languages including Chinese).

In [ ]:
import pytesseract
from sentence_transformers import SentenceTransformer

# Tesseract: chi_sim for simplified Chinese; chi_tra for traditional; jpn for Japanese.
# `+eng` lets it fall back on English when it sees Latin characters.
OCR_LANGS = "chi_sim+eng"

def ocr_pages(pil_pages):
    texts = []
    for i, p in enumerate(pil_pages):
        text = pytesseract.image_to_string(p, lang=OCR_LANGS)
        texts.append(text)
        if i < 2:
            preview = text.strip().replace("\n", " ")[:120]
            print(f"page {i}: {preview}...")
    return texts

print("Running OCR over all pages (this is the slow part of the traditional pipeline)...")
page_texts = ocr_pages(pages)

Look at those previews. Tesseract on Chinese has predictable failure modes:
- Numbers and English mixed inside Chinese text get garbled (`Q3` → `O3`, `2023年` → `2023 # `)
- Tables lose their column structure — every row becomes a line of space-separated tokens
- Vertical text or rotated text becomes salad
- Charts are read as random characters or skipped

**This is the data your downstream embedder is working with.** Garbage in, garbage out.

In [ ]:
# Chunk each page's OCR output into ~3 chunks (very simple — production would be smarter).
import re

def chunk_page_text(text: str, max_chars: int = 600):
    # Split on Chinese / English sentence boundaries, then pack into chunks.
    sentences = re.split(r"(?<=[。！？!?\.])\s+|\n\n+", text)
    chunks, buf = [], ""
    for s in sentences:
        if len(buf) + len(s) > max_chars and buf:
            chunks.append(buf.strip()); buf = ""
        buf += s + " "
    if buf.strip():
        chunks.append(buf.strip())
    return [c for c in chunks if len(c) > 30]   # drop tiny fragments

ocr_chunks = []   # list of (page_idx, chunk_text)
for page_idx, text in enumerate(page_texts):
    for c in chunk_page_text(text):
        ocr_chunks.append((page_idx, c))
print(f"Total OCR chunks: {len(ocr_chunks)}")

ST_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
st_model = SentenceTransformer(ST_NAME)
ocr_embs = st_model.encode(
    [c for _, c in ocr_chunks], normalize_embeddings=True, show_progress_bar=False
)
print(f"OCR chunk embedding shape: {ocr_embs.shape}")

In [ ]:
import numpy as np

def ocr_search(query: str, k: int = 3):
    q_emb = st_model.encode([query], normalize_embeddings=True)[0]
    sims = ocr_embs @ q_emb
    top_idx = np.argsort(-sims)[:k]
    return [(float(sims[i]), ocr_chunks[i][0], ocr_chunks[i][1]) for i in top_idx]

# Sanity check
for s, p, c in ocr_search("营业收入", k=2):
    print(f"score={s:.3f} page={p}\n  {c[:120]}...")

## Comparison — same questions, two retrieval paths

Three queries with different stress profiles. Adapt to your actual PDF:
1. **Generic prose query** — both should do fine.
2. **Chart/figure query** — ColPali should crush this; OCR baseline can't read chart bars.
3. **Specific number in a table** — ColPali should locate the cell; OCR may have garbled the table.

In [ ]:
QUERIES = [
    "公司业务概述",                # generic prose
    "营业收入趋势图",              # chart query
    "研发费用占营业收入的比例",   # table cell / specific metric
]

for q in QUERIES:
    print(f"\n=== Query: {q} ===")
    cp_hits = colpali_search(q, page_embeddings, k=3)
    print("  ColPali top pages:", [idx for _, idx in cp_hits])
    ocr_hits = ocr_search(q, k=3)
    print("  OCR     top pages:", [p for _, p, _ in ocr_hits])

In [ ]:
# Visualize side-by-side for one of the chart queries
CHART_Q = QUERIES[1]
cp_top = colpali_search(CHART_Q, page_embeddings, k=1)[0][1]
ocr_top = ocr_search(CHART_Q, k=1)[0][1]

fig, axes = plt.subplots(1, 2, figsize=(12, 7))
axes[0].imshow(pages[cp_top]); axes[0].set_title(f"ColPali → page {cp_top}"); axes[0].set_axis_off()
axes[1].imshow(pages[ocr_top]); axes[1].set_title(f"OCR baseline → page {ocr_top}"); axes[1].set_axis_off()
fig.suptitle(f"Query: '{CHART_Q}'", fontsize=12)
plt.tight_layout(); plt.show()

Look at the OCR baseline's answer page. Often it picks a **text-only page that mentions the keyword** (e.g., a paragraph that contains "营业收入") instead of the actual chart page. That's because the chart's axis labels and bar values either weren't extracted at all, or were extracted as garbled tokens that don't survive embedding.

## Send the winner to a VLM (closing the loop)

Same as nb03 / nb04: feed the top page to GPT-4o for the answer. GPT-4o handles Chinese natively, so we don't change the call shape — just the system prompt language.

In [ ]:
import base64, io, os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(dotenv_path="../.env")
client = OpenAI()

def page_to_b64(p, q=85):
    buf = io.BytesIO(); p.save(buf, format="JPEG", quality=q)
    return base64.b64encode(buf.getvalue()).decode()

SYSTEM_ZH = (
    "你是一名财务分析助手。仅根据提供的 PDF 页面图像回答用户问题，"
    "必须引用看到的具体内容（图表标题、表格数值、段落文字）。"
    "如果页面上没有相关信息，明确说明'页面未提供'。"
)

def answer_zh(query: str, page_idx: int) -> str:
    r = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_ZH},
            {"role": "user", "content": [
                {"type": "image_url", "image_url": {
                    "url": f"data:image/jpeg;base64,{page_to_b64(pages[page_idx])}",
                    "detail": "high",
                }},
                {"type": "text", "text": query},
            ]},
        ],
    )
    return r.choices[0].message.content

for q in QUERIES:
    cp_page = colpali_search(q, page_embeddings, k=1)[0][1]
    print(f"\n问: {q}  (page {cp_page})")
    print(f"答: {answer_zh(q, cp_page)}")

## What we learned

- **ColPali is multilingual for free.** Same model, same code, different PDF — the visual encoder doesn't care about the script. PaliGemma was pretrained on a multilingual corpus, and we never assumed English.
- **OCR pipelines pay a multilingual tax** that compounds: the OCR engine itself, layout post-processing, and the embedding model all need to handle each new language. Quality drops at every layer.
- **Charts and tables are where the gap is widest.** OCR can't read a chart bar's value. ColPali matches query tokens against the patch where the bar lives.

**Same idea applies to Japanese, Korean, Arabic, mixed-script documents.** Try the same notebook with `jpn` Tesseract lang on a 決算短信 — you'll see the OCR baseline degrade further (vertical text, kanji ambiguities) while ColPali holds.

## Where this goes next (Phase 3)

We've handled images and documents. Next: **audio**. The other half of multimodal. Same questions:
- How does a model "see" speech?
- What's the equivalent of the 'modality gap' for audio?
- And specifically: **how do you transcribe code-switched speech** (中英文混说) without the model silently translating one language into the other?

On to nb06.